# Part 3 · Challenge: you are the station manager
```
            6  City Exit (Y)
            |
            4  Mall Junction
            |  \
   (wide)   |    \   (narrow diagonal)
            |      \
   0 ───────2 ───────3 ─────────── 5  Platform (X)
  West     Plaza  ↑  Hall
  Gate (A)    NARROW  |
                      1  South Gate (B)
```
With signs, staff and app messages you can steer **which destination people from each gate head for**
(think *"Platform entrance via the South Gate"*). In model terms: **you choose the OD weights.**
The weights only *steer* people, so check what the simulation actually delivers.

### Rules
* Gate demand is fixed.
* The train company requires that **at least 60%** of everyone who reaches a destination arrives at the
  **Platform X**. This is measured in the **simulation**, not read off your weights.
* **Score = time spent inside (person-hours) + 0.25 h for every person turned away. Lower is better.**
* Post your best valid score on the leaderboard.

**How to use this notebook**
* Press **▶** on the left of a grey cell (or click it and press **Shift + Enter**) to run it.
* Cells with sliders and boxes are *forms*: change the values on the right, then press ▶ again.
* You never have to edit code. If you're curious, double-click a form to see the code behind it.
* Something broken? **Runtime → Restart session**, then run the first cell again.

## ▶ Step 1: install (run this first, ≈30 s)
This is how you install **any** Python package from GitHub. `pednstream-workshop` is a small
package with this workshop's network and plotting helpers; installing it also installs
**PedNStream** itself. If Colab restarted, run it again.
You can ignore the note *"you may need to restart the kernel"*.

In [ ]:
%pip install -q "pednstream-workshop @ git+https://github.com/WaimenMak/pednstream-workshop.git"
import json, importlib.metadata as md
if json.loads(md.distribution("pednstream").read_text("direct_url.json") or "{}").get("vcs_info", {}).get("commit_id") != "4a85236b3c47b17f25c2bfe8d5c0ef35651900a7":
    %pip install -q --force-reinstall --no-deps "pednstream @ git+https://github.com/WaimenMak/PedNStream.git@4a85236b3c47b17f25c2bfe8d5c0ef35651900a7"

import matplotlib.pyplot as plt
import pednstream_workshop
if tuple(map(int, pednstream_workshop.__version__.split("."))) < (1, 4, 0):
    raise RuntimeError("An old version of the workshop package is loaded. "
                       "Click Runtime → Restart session, then run this cell again.")
from pednstream_workshop import run, compare, compare_maps, AMSTERDAM, arrival_plan, widen
plt.rcParams["figure.dpi"] = 90
print("✅ Ready! Scroll down and continue.")

In [ ]:
#@title 🏆 **Your strategy**: set the values, then press ▶ {display-mode: "form"}
team_name = "my team"  #@param {type:"string"}
#@markdown Weight towards the Platform per gate (the rest of the weight goes to the City Exit)
weight_Gate_A_to_Platform = 0.6  #@param {type:"slider", min:0, max:1, step:0.05}
weight_Gate_B_to_Platform = 0.6  #@param {type:"slider", min:0, max:1, step:0.05}
#@markdown **Bonus round only**: widen one corridor by 2 m (leave at "none" for the main round)
upgrade = "none"  #@param ["none", "0_2", "1_3", "2_3", "2_4", "3_4", "3_5", "4_6"]

a, b = weight_Gate_A_to_Platform, weight_Gate_B_to_Platform
links = {}
if upgrade != "none":
    base_width = run().link(upgrade).width
    links = {upgrade: {"width": base_width + 2}}
res = run(od={"0_5": a, "0_6": 1 - a, "1_5": b, "1_6": 1 - b}, links=links)
k = res.kpis()
score = k["time spent inside (person-hours)"] + 0.25 * k["turned away at gates"]
to_X = k["reached Platform X"] / (k["reached Platform X"] + k["reached City Exit Y"])
valid = to_X >= 0.6

if "attempts" not in globals():
    attempts = []
attempts.append({"team": team_name, "A→X": a, "B→X": b, "upgrade": upgrade,
                 "delivered to X": f"{to_X:.1%}", "valid": "✅" if valid else "❌", "score": round(float(score), 1)})

print(("✅ Valid strategy" if valid else "❌ BREAKS the 60% rule: this score does not count") +
      f"  ({to_X:.1%} of arrivals reached the Platform; weights asked for {(a + b) / 2:.0%})")
print(f"\n   SCORE = {score:.1f}     (time inside {k['time spent inside (person-hours)']:.1f} h, turned away {k['turned away at gates']})\n")
if valid:
    print(f"📋 Leaderboard line:  {team_name} | A→X {a:.2f} | B→X {b:.2f} | upgrade {upgrade} | score {score:.1f}")
res.dashboard(f"{team_name}: A→X {a:.0%}, B→X {b:.0%}"); plt.show()
import pandas as pd
print("Your attempts so far (best first):")
display(pd.DataFrame(attempts).sort_values(["valid", "score"], ascending=[False, True]).reset_index(drop=True))

### Stuck? Hints (open one at a time)
<details><summary>Hint 1</summary>Look at the map of your strategy: which corridors are red, and which OD groups walk through them?</details>
<details><summary>Hint 2</summary>Gate B has only <b>one</b> route to the Platform, and it shares nothing with Gate A's routes.</details>
<details><summary>Hint 3</summary>Remember Exercise 2: the crossing pattern was expensive, the parallel one was cheap.</details>
<details><summary>Hint 4</summary>Just missed 60%? The weights steer, the simulation decides. Nudge one weight up a little.</details>

---
### For coders (optional): search systematically
Instead of guessing, scan every combination and draw a heat-map of the score. Edit and run the cell below.

In [ ]:
import numpy as np
shares = np.round(np.arange(0, 1.01, 0.1), 1)
scores = np.full((len(shares), len(shares)), np.nan)
for i, a in enumerate(shares):
    for j, b in enumerate(shares):
        if (a + b) / 2 < 0.5:
            continue                       # far too few to X, skip to save time
        k = run(od={"0_5": a, "0_6": 1 - a, "1_5": b, "1_6": 1 - b}).kpis()
        if k["reached Platform X"] / (k["reached Platform X"] + k["reached City Exit Y"]) < 0.6:
            continue                       # breaks the 60% rule
        scores[i, j] = k["time spent inside (person-hours)"] + 0.25 * k["turned away at gates"]

plt.imshow(scores, origin="lower", cmap="viridis_r")
plt.xticks(range(len(shares)), shares); plt.yticks(range(len(shares)), shares)
plt.xlabel("Gate B weight to Platform"); plt.ylabel("Gate A weight to Platform")
plt.colorbar(label="score (lower is better)"); plt.title("Challenge score"); plt.show()
i, j = np.unravel_index(np.nanargmin(scores), scores.shape)
print(f"Best on this grid: A→X {shares[i]}, B→X {shares[j]}, score {scores[i, j]:.1f}")

### Reflection
1. Describe your winning strategy in words a station manager would understand.
2. Why does it work? Name specific corridors.
3. In the bonus round, did one wider corridor help more or less than a smart OD strategy?
4. Name one thing the model leaves out that could make your strategy fail in real life.

---
# Challenge 2 · Event day: from Centraal Station to Museumplein
```
  2 Westermarkt ---Nieuwendijk/Leliegracht--- 9 Stationsplein --- 0 CENTRAAL STATION (start)
  (Anne Frank)  \                           /         \
     |       Raadhuisstraat           DAMRAK            Zeedijk
     |         (3 m, trams)        (20 m wide!)            \
     |                 \               /                   3 Nieuwmarkt
  Prinsen-              1 Dam (Madame Tussauds)              |  Jodenbreestraat
  gracht                |                                    5 Waterlooplein (Rembrandthuis)
     |            Kalverstraat (3 m)              Amstel   / |
     |                  |                 ---------------    | Utrechtsestraat
     |                  4 Muntplein                          |
     |   Leidsestraat /   \ Vijzelstraat                      |
     6 Leidseplein          10 Weteringcircuit --------------+  Weteringschans
       |                     |  Stadhouderskade
       |  Hobbemastraat      7 Rijksmuseum
        \                  /  passage
          8 MUSEUMPLEIN (event)
```
The numbers are the main places. On the maps you'll also see small grey dots (11, 12, …): points along
each street, about 300 m apart, so you can see *where* on a street a crowd builds up.
A big open-air event starts on **Museumplein** and the trains bring thousands of visitors into
**Amsterdam Centraal**. Stewards on the platforms decide **how many people per minute** are let out of
the station towards the event. That is your lever: **you design the arrival pattern** for the first hour.

The obvious way out is **Damrak**, past Madame Tussauds on the Dam: a *very* wide street (20 m),
so it looks like it can take anyone. But at the Dam everyone has to squeeze into the narrow
**Kalverstraat** or the tram-filled **Raadhuisstraat** (both 3 m of usable width). People choose their own
route (shorter, wider and emptier streets are more attractive), just like in Part 1.

The streets, their lengths and the points along them come from **OpenStreetMap**, so the network
follows the real city.

### Rules
* You set the number of people per minute leaving the station for six 10-minute blocks (minute 0 to 60).
* The simulation runs for **2 hours**. A walk on empty streets takes about **53 minutes** (about 3.5 km).
* **Score = people who reach Museumplein − hours that people spend on the streets (person-hours).**
  Every visitor you deliver earns 1 point; every hour someone spends walking or waiting costs 1 point.
  So a visitor whose trip takes longer than an hour costs more than they bring in. With a 53-minute walk
  there is not much room for queueing! **Higher is better.**
* Post your best score on the leaderboard.

## ▶ Step 2: load the Amsterdam tools (if you skipped Challenge 1, run Step 1 first)

In [ ]:
import pandas as pd
from pednstream_workshop import AMSTERDAM, arrival_plan, widen
print("✅ Amsterdam loaded.")

In [ ]:
#@title 🎟️ **Your arrival plan**: set the values, then press ▶ {display-mode: "form"}
team_name = "my team"  #@param {type:"string"}
#@markdown People per minute let out of Centraal Station, for each 10-minute block of the first hour
minutes_00_10 = 300  #@param {type:"slider", min:0, max:900, step:30}
minutes_10_20 = 300  #@param {type:"slider", min:0, max:900, step:30}
minutes_20_30 = 300  #@param {type:"slider", min:0, max:900, step:30}
minutes_30_40 = 300  #@param {type:"slider", min:0, max:900, step:30}
minutes_40_50 = 300  #@param {type:"slider", min:0, max:900, step:30}
minutes_50_60 = 300  #@param {type:"slider", min:0, max:900, step:30}
#@markdown **Bonus round only**: widen one street by 2 m (leave at "none" for the main round)
upgrade = "none"  #@param ["none", "Damrak", "Kalverstraat", "Raadhuisstraat", "Prinsengracht", "Leidsestraat", "Vijzelstraat", "Zeedijk", "Nieuwendijk / Leliegracht"]

plan = [minutes_00_10, minutes_10_20, minutes_20_30, minutes_30_40, minutes_40_50, minutes_50_60]
links = widen(upgrade, 2) if upgrade != "none" else {}   # +2 m along the whole street
res = run(network_dir=AMSTERDAM, arrivals={0: arrival_plan(plan)}, links=links)
last_plan = res   # remembered for the animation below
k = res.kpis()
sent, arrived = k["total demand (people)"], k["reached Museumplein"]
hours = float(k["time spent inside (person-hours)"])
score = arrived - hours

if "plans" not in globals():
    plans = []
plans.append({"team": team_name, **{f"{10 * i}-{10 * i + 10}'": p for i, p in enumerate(plan)},
              "upgrade": upgrade, "sent": sent, "arrived": arrived,
              "person-hours": round(hours), "score": round(score)})

print(f"🚆 Sent {sent:,} people   🎨 {arrived:,} reached Museumplein   ⏱ {hours:,.0f} person-hours on the streets")
print(f"   average trip: {60 * hours / max(sent, 1):.0f} min (empty streets: about 53 min)")
print(f"\n   SCORE = {score:,.0f}\n")
print(f"📋 Leaderboard line:  {team_name} | plan {plan} | upgrade {upgrade} | score {score:.0f}")
res.dashboard(f"{team_name}: {plan} people/min", flow=True); plt.show()
print("Your plans so far (best first):")
display(pd.DataFrame(plans).sort_values("score", ascending=False).reset_index(drop=True))

### 🎬 Watch your plan play out
Run your plan above first, then this cell. It replays the **last plan you ran**: the map shows how crowded
each street is, minute by minute, and the black line on the right shows where you are in time.
Press ▶ under the animation; use − / + to slow down or speed up. (Takes ≈30 s to draw.)

In [ ]:
#@title 🎬 Animate your last plan {display-mode: "form"}
#@markdown Show one frame every ... simulated minutes (smaller = smoother but slower to draw)
minutes_per_frame = 2  #@param {type:"slider", min:0.5, max:5, step:0.5}
if "last_plan" not in globals():
    print("⚠️ Run your arrival plan (the 🎟️ cell above) first.")
else:
    display(last_plan.animate(every=max(1, round(minutes_per_frame * 6)), flow=True))

### Stuck? Hints (open one at a time)
<details><summary>Hint 1</summary>Look at the <i>In and out</i> plot. Does the red line (arrivals per minute)
follow your blue blocks, about 53 minutes later? When does it stop following them?</details>
<details><summary>Hint 2</summary>When too many people arrive at once, they wait somewhere. Where? Watch the
animation: which street fills up first, and where does the queue go next? Is it the wide Damrak?</details>
<details><summary>Hint 3</summary>Someone who leaves at minute 55 needs about 53 minutes on empty streets.
Do they make it before the end (minute 120)? What if they have to queue?</details>
<details><summary>Hint 4</summary>Your blocks don't have to be equal. A bigger group is fine as long as
the queue it causes clears quickly. Is there a moment in the hour when nobody comes after them?</details>

---
### For coders (optional): let the computer search
A simple *hill climb*: start from a plan, try moving each block up or down, keep any change that improves
the score, and repeat with smaller steps. (≈2–3 minutes.) Can you beat it by hand? Can you write a better search?

In [ ]:
def amsterdam_score(plan):
    k = run(network_dir=AMSTERDAM, arrivals={0: arrival_plan(plan)}).kpis()
    return k["reached Museumplein"] - float(k["time spent inside (person-hours)"])

best = [300] * 6
best_score = amsterdam_score(best)
for step in [120, 60, 30]:
    improved = True
    while improved:
        improved = False
        for i in range(6):
            for change in (-step, step):
                trial = best.copy()
                trial[i] = min(900, max(0, trial[i] + change))
                s = amsterdam_score(trial)
                if s > best_score + 1:
                    best, best_score, improved = trial, s, True
    print(f"step {step:3d}: best plan {best}, score {best_score:,.0f}")

### Reflection
1. Describe your arrival plan in words a steward on the platform would understand.
2. Why does it work? Name the street that decides how many people per minute can reach Museumplein.
3. Damrak is 20 m wide. Did that help or hurt? What does a very wide street do to *where* people choose to walk?
4. Almost nobody takes Zeedijk and the eastern route past Waterlooplein, even when Kalverstraat is packed. Why might the
   model's route choice do that, and would real visitors behave the same way?
5. In the bonus round, which street was worth widening, and was it worth more than a smart plan?